In [ ]:
# | default_exp retrieval.layout_rag

# Hybrid retrieval for layout-aware OCR

Idempotent canonical storage, dense/lexical fusion, evidence expansion, and citations.

In [ ]:
# | export
import inspect
import hashlib
import json
import math
import re
import sqlite3
import unicodedata
from collections import defaultdict
from contextlib import nullcontext
from dataclasses import asdict, dataclass, field
from pathlib import Path
from typing import Any, Awaitable, Callable, Mapping, Protocol, Sequence

from ribosome.preprocessing.ocr.layout_bundle import (
    COMMON_IDENTIFIER_ALIASES,
    LAYOUT_PIPELINE_VERSION,
    LayoutDocument,
    LayoutIngestionResult,
    LayoutRegion,
    LayoutRetrievalRecord,
    LayoutSection,
)


_QUERY_IDENTIFIER_RE = re.compile(
    r"(?<![A-Za-z0-9_])([A-Za-z][A-Za-z0-9_]*(?:\[\])?(?:\.[A-Za-z]+)?)(?![A-Za-z0-9_])"
)
_SYNTAX_QUERY_RE = re.compile(r"[A-Z][A-Z0-9_]{2,}|\[\]|\[[0-9]+\]|[.=<>]|\b(?:IN|IO)\b")
_VISUAL_QUERY_RE = re.compile(r"(?:图|图示|示意|外观|布局|figure|diagram|image)", re.IGNORECASE)
_SQLITE_SCHEMA_VERSION = 2
_DEFAULT_MAX_QUERY_CHARACTERS = 2048
_DEFAULT_MAX_QUERY_TERMS = 128


def _json(value: Any) -> str:
    return json.dumps(value, ensure_ascii=False, sort_keys=True, separators=(",", ":"))


def _normalized_term(value: str) -> str:
    return " ".join(unicodedata.normalize("NFKC", value).casefold().split())


def _record_from_json(value: str) -> LayoutRetrievalRecord:
    payload = json.loads(value)
    payload.setdefault("canonical_texts", tuple(payload.get("cleaned_ocr_texts") or ()))
    payload.setdefault("region_quality_flags", ())
    payload.setdefault("region_asset_paths", tuple(payload.get("asset_paths") or ()))
    payload.setdefault("source_pdf_uri", None)
    payload.setdefault("source_markdown_uri", "")
    payload.setdefault("source_layout_uri", "")
    payload.setdefault("source_hashes", ())
    payload.setdefault("pipeline_version", LAYOUT_PIPELINE_VERSION)
    for key in (
        "region_ids",
        "bboxes",
        "bboxes_normalized",
        "region_labels",
        "ocr_statuses",
        "quality_flags",
        "quality_reasons",
        "text_sources",
        "asset_paths",
        "aliases",
        "canonical_texts",
        "cleaned_ocr_texts",
        "native_texts",
        "detector_scores",
        "source_hashes",
        "region_asset_paths",
    ):
        payload[key] = tuple(tuple(item) if isinstance(item, list) else item for item in payload.get(key, ()))
    payload["region_quality_flags"] = tuple(
        tuple(dict(flag) for flag in flags)
        for flags in payload.get("region_quality_flags", ())
    )
    return LayoutRetrievalRecord(**payload)


def _query_terms(
    query: str,
    *,
    max_characters: int = _DEFAULT_MAX_QUERY_CHARACTERS,
    max_terms: int = _DEFAULT_MAX_QUERY_TERMS,
) -> tuple[str, ...]:
    normalized = unicodedata.normalize("NFKC", query)[:max_characters]
    terms = {_normalized_term(normalized)}
    terms.update(_normalized_term(match) for match in _QUERY_IDENTIFIER_RE.findall(normalized))
    compact_cjk = "".join(character for character in normalized if "\u3400" <= character <= "\u9fff")
    terms.update(compact_cjk[index : index + 2] for index in range(max(0, len(compact_cjk) - 1)))
    ordered = sorted((term for term in terms if term), key=lambda term: (term != _normalized_term(normalized), term))
    return tuple(ordered[:max_terms])


def _looks_like_identifier(value: str) -> bool:
    return (
        value == value.upper()
        or any(character in value for character in "_[]")
        or any(character.isdigit() for character in value)
    )


def _record_terms(record: LayoutRetrievalRecord) -> tuple[tuple[str, str, str], ...]:
    terms: dict[str, tuple[str, str]] = {}
    for alias in record.aliases:
        normalized = _normalized_term(alias)
        canonical = COMMON_IDENTIFIER_ALIASES.get(alias.upper(), alias)
        terms[normalized] = (canonical, "instruction identifier or OCR alias")
    if record.instruction_code:
        terms[_normalized_term(record.instruction_code)] = (
            record.instruction_code,
            "instruction code",
        )
    for identifier in _QUERY_IDENTIFIER_RE.findall(record.exact_text):
        if not _looks_like_identifier(identifier):
            continue
        canonical = COMMON_IDENTIFIER_ALIASES.get(identifier.upper(), identifier)
        terms[_normalized_term(identifier)] = (canonical, "exact source identifier")
    compact_cjk = "".join(character for character in record.exact_text if "\u3400" <= character <= "\u9fff")
    for index in range(max(0, len(compact_cjk) - 1)):
        bigram = compact_cjk[index : index + 2]
        terms.setdefault(bigram, (bigram, "CJK bigram"))
    return tuple((term, canonical, reason) for term, (canonical, reason) in sorted(terms.items()) if term)

In [ ]:
# | export
@dataclass(frozen=True)
class LayoutRAGConfig:
    pipeline_version: str = LAYOUT_PIPELINE_VERSION
    lexical_top_k: int = 30
    dense_child_top_k: int = 30
    dense_parent_top_k: int = 15
    visual_top_k: int = 10
    rerank_top_k: int = 25
    result_limit: int = 8
    neighbor_radius: int = 1
    rrf_k: int = 60
    lexical_weight: float = 1.0
    dense_child_weight: float = 1.0
    dense_parent_weight: float = 0.7
    visual_weight: float = 0.6
    syntax_lexical_boost: float = 1.6
    max_hits_per_parent: int = 2
    max_hits_per_page: int = 2
    text_query_figure_weight: float = 0.55
    max_query_characters: int = _DEFAULT_MAX_QUERY_CHARACTERS
    max_query_terms: int = _DEFAULT_MAX_QUERY_TERMS
    max_visual_chunks_per_asset: int = 2

    def __post_init__(self) -> None:
        nonnegative_integers = {
            "lexical_top_k": self.lexical_top_k,
            "dense_child_top_k": self.dense_child_top_k,
            "dense_parent_top_k": self.dense_parent_top_k,
            "visual_top_k": self.visual_top_k,
            "rerank_top_k": self.rerank_top_k,
            "result_limit": self.result_limit,
            "neighbor_radius": self.neighbor_radius,
            "max_hits_per_parent": self.max_hits_per_parent,
            "max_hits_per_page": self.max_hits_per_page,
            "max_visual_chunks_per_asset": self.max_visual_chunks_per_asset,
        }
        invalid = [name for name, value in nonnegative_integers.items() if not isinstance(value, int) or value < 0]
        if invalid:
            raise ValueError(f"configuration values must be nonnegative integers: {', '.join(invalid)}")
        if not isinstance(self.rrf_k, int) or self.rrf_k <= 0:
            raise ValueError("rrf_k must be a positive integer")
        if self.max_query_characters <= 0 or self.max_query_terms <= 0:
            raise ValueError("query bounds must be positive")
        weights = {
            "lexical_weight": self.lexical_weight,
            "dense_child_weight": self.dense_child_weight,
            "dense_parent_weight": self.dense_parent_weight,
            "visual_weight": self.visual_weight,
            "syntax_lexical_boost": self.syntax_lexical_boost,
            "text_query_figure_weight": self.text_query_figure_weight,
        }
        invalid_weights = [
            name
            for name, value in weights.items()
            if not isinstance(value, (int, float)) or not math.isfinite(float(value)) or value < 0
        ]
        if invalid_weights:
            raise ValueError(f"configuration weights must be finite and nonnegative: {', '.join(invalid_weights)}")


@dataclass
class SearchHit:
    record: LayoutRetrievalRecord
    score: float
    channel_ranks: dict[str, int] = field(default_factory=dict)
    rerank_score: float | None = None


@dataclass(frozen=True)
class Evidence:
    hit: SearchHit
    parent: LayoutSection | None
    records: tuple[LayoutRetrievalRecord, ...]
    citation: str

    @property
    def exact_text(self) -> str:
        return "\n\n".join(record.exact_text for record in self.records)


@dataclass(frozen=True)
class RankedIdentifier:
    identifier: str
    score: float


class DenseCandidateIndex(Protocol):
    def query_children(self, query_embedding: Sequence[float], *, limit: int) -> tuple[tuple[str, float], ...]: ...
    def query_parents(self, query_embedding: Sequence[float], *, limit: int) -> tuple[tuple[str, float], ...]: ...


Reranker = Callable[[str, Sequence[SearchHit]], Sequence[SearchHit] | Mapping[str, float]]
VisualRetriever = Callable[[str, int], Sequence[str] | Sequence[tuple[str, float]]]
EmbeddingProvider = Callable[[Sequence[str]], Sequence[Sequence[float]] | Awaitable[Sequence[Sequence[float]]]]

In [ ]:
# | export
def _validate_ingestion_result(result: LayoutIngestionResult) -> None:
    """Reject cross-document/version references before either index is mutated."""
    document = result.document
    regions = document.regions
    region_ids = [region.region_id for region in regions]
    if len(region_ids) != len(set(region_ids)):
        raise ValueError("ingestion result contains duplicate region IDs")
    if any(region.document_id != document.document_id for region in regions):
        raise ValueError("every region must belong to the result document")

    parent_ids = [parent.parent_id for parent in result.sections]
    parent_id_set = set(parent_ids)
    if len(parent_ids) != len(parent_id_set):
        raise ValueError("ingestion result contains duplicate parent IDs")
    if any(parent.document_id != document.document_id for parent in result.sections):
        raise ValueError("every parent must belong to the result document")
    if any(
        parent.parent_section_id is not None and parent.parent_section_id not in parent_id_set
        for parent in result.sections
    ):
        raise ValueError("a parent references a section outside the result")

    region_id_set = set(region_ids)
    if any(region.parent_id not in parent_id_set for region in regions):
        raise ValueError("a region references a parent outside the result")
    if any(not set(parent.region_ids) <= region_id_set for parent in result.sections):
        raise ValueError("a parent references a region outside the result")
    chunk_ids: set[str] = set()
    for record in result.records:
        if record.chunk_id in chunk_ids:
            raise ValueError(f"duplicate chunk ID {record.chunk_id!r}")
        chunk_ids.add(record.chunk_id)
        if record.document_id != document.document_id:
            raise ValueError("every retrieval record must belong to the result document")
        if record.pipeline_version != document.pipeline_version:
            raise ValueError("every retrieval record must use the result pipeline version")
        if record.parent_id not in parent_id_set:
            raise ValueError(f"retrieval record references unknown parent {record.parent_id!r}")
        if not set(record.region_ids) <= region_id_set:
            raise ValueError(f"retrieval record {record.chunk_id!r} references unknown regions")
        if record.page_start <= 0 or record.page_end < record.page_start:
            raise ValueError(f"retrieval record {record.chunk_id!r} has an invalid page range")


class SQLiteRetrievalRecordStore:
    """Canonical graph plus FTS5/exact indexes with atomic document replacement."""

    def __init__(self, path: str | Path = ":memory:"):
        self.path = str(path)
        self.connection = sqlite3.connect(self.path)
        self.connection.row_factory = sqlite3.Row
        self.connection.execute("PRAGMA foreign_keys = ON")
        if self.path != ":memory:":
            self.connection.execute("PRAGMA journal_mode = WAL")
        self._create_schema()

    def _create_schema(self) -> None:
        self.connection.executescript(
            """
            CREATE TABLE IF NOT EXISTS documents (
                document_id TEXT NOT NULL,
                pipeline_version TEXT NOT NULL,
                title TEXT NOT NULL,
                document_code TEXT NOT NULL,
                revision TEXT NOT NULL,
                status TEXT NOT NULL,
                schema_version INTEGER NOT NULL,
                artifact_hashes_json TEXT NOT NULL,
                metadata_json TEXT NOT NULL,
                PRIMARY KEY (document_id, pipeline_version)
            );
            CREATE TABLE IF NOT EXISTS pages (
                document_id TEXT NOT NULL,
                pipeline_version TEXT NOT NULL,
                page_number INTEGER NOT NULL,
                width INTEGER NOT NULL,
                height INTEGER NOT NULL,
                status TEXT NOT NULL,
                error TEXT,
                PRIMARY KEY (document_id, pipeline_version, page_number)
            );
            CREATE TABLE IF NOT EXISTS regions (
                region_id TEXT NOT NULL,
                document_id TEXT NOT NULL,
                pipeline_version TEXT NOT NULL,
                page_number INTEGER NOT NULL,
                region_index INTEGER NOT NULL,
                parent_id TEXT NOT NULL,
                label TEXT NOT NULL,
                task_type TEXT NOT NULL,
                ocr_status TEXT NOT NULL,
                bbox_json TEXT NOT NULL,
                bbox_normalized_json TEXT NOT NULL,
                canonical_text TEXT NOT NULL,
                ocr_content TEXT NOT NULL,
                native_text TEXT NOT NULL,
                raw_content TEXT NOT NULL,
                raw_markdown TEXT NOT NULL,
                text_source TEXT NOT NULL,
                quality_flags_json TEXT NOT NULL,
                quarantined INTEGER NOT NULL,
                asset_path TEXT,
                previous_region_id TEXT,
                next_region_id TEXT,
                provenance_json TEXT NOT NULL,
                PRIMARY KEY (region_id, pipeline_version)
            );
            CREATE TABLE IF NOT EXISTS parents (
                parent_id TEXT NOT NULL,
                document_id TEXT NOT NULL,
                pipeline_version TEXT NOT NULL,
                number TEXT,
                title TEXT NOT NULL,
                level INTEGER NOT NULL,
                section_path TEXT NOT NULL,
                parent_section_id TEXT,
                page_start INTEGER NOT NULL,
                page_end INTEGER NOT NULL,
                region_ids_json TEXT NOT NULL,
                summary TEXT NOT NULL,
                instruction_code TEXT,
                is_toc INTEGER NOT NULL,
                record_json TEXT NOT NULL,
                PRIMARY KEY (parent_id, pipeline_version)
            );
            CREATE TABLE IF NOT EXISTS chunks (
                chunk_id TEXT PRIMARY KEY,
                document_id TEXT NOT NULL,
                pipeline_version TEXT NOT NULL,
                parent_id TEXT NOT NULL,
                content_type TEXT NOT NULL,
                page_start INTEGER NOT NULL,
                page_end INTEGER NOT NULL,
                reading_order INTEGER NOT NULL,
                indexable INTEGER NOT NULL,
                retrieval_text TEXT NOT NULL,
                exact_text TEXT NOT NULL,
                instruction_code TEXT,
                section_path TEXT NOT NULL,
                record_json TEXT NOT NULL
            );
            CREATE INDEX IF NOT EXISTS chunks_document_version
                ON chunks(document_id, pipeline_version);
            CREATE INDEX IF NOT EXISTS chunks_parent_order
                ON chunks(parent_id, reading_order);
            CREATE TABLE IF NOT EXISTS chunk_regions (
                chunk_id TEXT NOT NULL,
                region_id TEXT NOT NULL,
                ordinal INTEGER NOT NULL,
                PRIMARY KEY (chunk_id, region_id)
            );
            CREATE TABLE IF NOT EXISTS exact_terms (
                term TEXT NOT NULL,
                chunk_id TEXT NOT NULL,
                canonical_term TEXT NOT NULL,
                reason TEXT NOT NULL,
                PRIMARY KEY (term, chunk_id)
            );
            CREATE INDEX IF NOT EXISTS exact_terms_term ON exact_terms(term);
            """
        )
        rebuilt_fts: set[str] = set()
        expected_fts_columns = {
            "chunks_fts": {"chunk_id", "retrieval_text", "section_path", "instruction_code", "aliases"},
            "parents_fts": {"parent_id", "pipeline_version", "summary", "section_path", "instruction_code"},
        }
        for table, expected in expected_fts_columns.items():
            columns = {row[1] for row in self.connection.execute(f"PRAGMA table_info({table})")}
            if columns and columns != expected:
                self.connection.execute(f"DROP TABLE {table}")
                rebuilt_fts.add(table)
        try:
            self.connection.execute(
                "CREATE VIRTUAL TABLE IF NOT EXISTS chunks_fts USING fts5("
                "chunk_id UNINDEXED, retrieval_text, section_path, instruction_code, aliases, tokenize='trigram')"
            )
            self.connection.execute(
                "CREATE VIRTUAL TABLE IF NOT EXISTS parents_fts USING fts5("
                "parent_id UNINDEXED, pipeline_version UNINDEXED, summary, section_path, instruction_code, tokenize='trigram')"
            )
        except sqlite3.OperationalError:
            self.connection.execute(
                "CREATE VIRTUAL TABLE IF NOT EXISTS chunks_fts USING fts5("
                "chunk_id UNINDEXED, retrieval_text, section_path, instruction_code, aliases, tokenize='unicode61')"
            )
            self.connection.execute(
                "CREATE VIRTUAL TABLE IF NOT EXISTS parents_fts USING fts5("
                "parent_id UNINDEXED, pipeline_version UNINDEXED, summary, section_path, instruction_code, tokenize='unicode61')"
            )
        if "chunks_fts" in rebuilt_fts:
            self.connection.execute(
                "INSERT INTO chunks_fts(chunk_id,retrieval_text,section_path,instruction_code,aliases) "
                "SELECT chunk_id,retrieval_text,section_path,coalesce(instruction_code,''),'' FROM chunks WHERE indexable=1"
            )
        if "parents_fts" in rebuilt_fts:
            self.connection.execute(
                "INSERT INTO parents_fts(parent_id,pipeline_version,summary,section_path,instruction_code) "
                "SELECT parent_id,pipeline_version,summary,section_path,coalesce(instruction_code,'') "
                "FROM parents WHERE summary<>'' AND is_toc=0"
            )
        self.connection.execute(f"PRAGMA user_version = {_SQLITE_SCHEMA_VERSION}")
        self.connection.commit()

    @staticmethod
    def _document_metadata(document: LayoutDocument) -> dict[str, Any]:
        return {
            "provider": document.provider,
            "ocr_model": document.ocr_model,
            "layout_model": document.layout_model,
            "dpi": document.dpi,
            "layout_path": str(document.paths.layout_path),
            "markdown_path": str(document.paths.markdown_path),
            "pdf_path": str(document.paths.pdf_path) if document.paths.pdf_path else None,
        }

    @staticmethod
    def _region_flags(region: LayoutRegion) -> list[dict[str, str]]:
        return [asdict(flag) for flag in region.quality_flags]

    def _delete_version(self, document_id: str, pipeline_version: str) -> None:
        chunk_ids = [
            row[0]
            for row in self.connection.execute(
                "SELECT chunk_id FROM chunks WHERE document_id=? AND pipeline_version=?",
                (document_id, pipeline_version),
            )
        ]
        parent_ids = [
            row[0]
            for row in self.connection.execute(
                "SELECT parent_id FROM parents WHERE document_id=? AND pipeline_version=?",
                (document_id, pipeline_version),
            )
        ]
        for chunk_id in chunk_ids:
            self.connection.execute("DELETE FROM chunks_fts WHERE chunk_id=?", (chunk_id,))
        for parent_id in parent_ids:
            self.connection.execute(
                "DELETE FROM parents_fts WHERE parent_id=? AND pipeline_version=?",
                (parent_id, pipeline_version),
            )
        if chunk_ids:
            placeholders = ",".join("?" for _ in chunk_ids)
            self.connection.execute(f"DELETE FROM exact_terms WHERE chunk_id IN ({placeholders})", chunk_ids)
            self.connection.execute(f"DELETE FROM chunk_regions WHERE chunk_id IN ({placeholders})", chunk_ids)
        self.connection.execute("DELETE FROM chunks WHERE document_id=? AND pipeline_version=?", (document_id, pipeline_version))
        self.connection.execute("DELETE FROM parents WHERE document_id=? AND pipeline_version=?", (document_id, pipeline_version))
        self.connection.execute("DELETE FROM regions WHERE document_id=? AND pipeline_version=?", (document_id, pipeline_version))
        self.connection.execute("DELETE FROM pages WHERE document_id=? AND pipeline_version=?", (document_id, pipeline_version))
        self.connection.execute("DELETE FROM documents WHERE document_id=? AND pipeline_version=?", (document_id, pipeline_version))

    def replace(self, result: LayoutIngestionResult, *, commit: bool = True) -> None:
        """Atomically replace one document/pipeline version, including stale FTS rows."""
        _validate_ingestion_result(result)
        document = result.document
        version = document.pipeline_version
        with (self.connection if commit else nullcontext()):
            self._delete_version(document.document_id, version)
            self.connection.execute(
                "INSERT INTO documents VALUES (?,?,?,?,?,?,?,?,?)",
                (
                    document.document_id,
                    version,
                    document.document_title,
                    document.document_code,
                    document.revision,
                    document.status,
                    document.schema_version,
                    _json(document.artifact_hashes),
                    _json(self._document_metadata(document)),
                ),
            )
            for page in document.pages:
                self.connection.execute(
                    "INSERT INTO pages VALUES (?,?,?,?,?,?,?)",
                    (document.document_id, version, page.page_number, page.width, page.height, page.status, page.error),
                )
            for region in document.regions:
                self.connection.execute(
                    "INSERT INTO regions VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)",
                    (
                        region.region_id,
                        document.document_id,
                        version,
                        region.page_number,
                        region.region_index,
                        region.parent_id,
                        region.label,
                        region.task_type,
                        region.ocr_status,
                        _json(region.bbox.as_tuple()),
                        _json(region.bbox_normalized),
                        region.canonical_text,
                        region.ocr_content,
                        region.native_text,
                        region.raw_content,
                        region.raw_markdown,
                        region.text_source,
                        _json(self._region_flags(region)),
                        int(region.quarantined),
                        str(region.asset_path) if region.asset_path else None,
                        region.previous_region_id,
                        region.next_region_id,
                        _json(region.provenance),
                    ),
                )
            for parent in result.sections:
                parent_payload = asdict(parent)
                self.connection.execute(
                    "INSERT INTO parents VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?,?)",
                    (
                        parent.parent_id,
                        document.document_id,
                        version,
                        parent.number,
                        parent.title,
                        parent.level,
                        parent.section_path,
                        parent.parent_section_id,
                        parent.page_start,
                        parent.page_end,
                        _json(parent.region_ids),
                        parent.summary,
                        parent.instruction_code,
                        int(parent.is_toc),
                        _json(parent_payload),
                    ),
                )
                if parent.summary and not parent.is_toc:
                    self.connection.execute(
                        "INSERT INTO parents_fts(parent_id,pipeline_version,summary,section_path,instruction_code) VALUES (?,?,?,?,?)",
                        (parent.parent_id, version, parent.summary, parent.section_path, parent.instruction_code or ""),
                    )
            for record in result.records:
                self.connection.execute(
                    "INSERT INTO chunks VALUES (?,?,?,?,?,?,?,?,?,?,?,?,?,?)",
                    (
                        record.chunk_id,
                        record.document_id,
                        record.pipeline_version,
                        record.parent_id,
                        record.content_type,
                        record.page_start,
                        record.page_end,
                        record.reading_order,
                        int(record.indexable),
                        record.retrieval_text,
                        record.exact_text,
                        record.instruction_code,
                        record.section_path,
                        _json(record.to_dict()),
                    ),
                )
                for ordinal, region_id in enumerate(record.region_ids):
                    self.connection.execute("INSERT INTO chunk_regions VALUES (?,?,?)", (record.chunk_id, region_id, ordinal))
                for term, canonical, reason in _record_terms(record):
                    self.connection.execute(
                        "INSERT OR IGNORE INTO exact_terms VALUES (?,?,?,?)",
                        (term, record.chunk_id, canonical, reason),
                    )
                if record.indexable:
                    self.connection.execute(
                        "INSERT INTO chunks_fts(chunk_id,retrieval_text,section_path,instruction_code,aliases) VALUES (?,?,?,?,?)",
                        (record.chunk_id, record.retrieval_text, record.section_path, record.instruction_code or "", " ".join(record.aliases)),
                    )

    def delete_document(self, document_id: str, *, pipeline_version: str = LAYOUT_PIPELINE_VERSION) -> None:
        with self.connection:
            self._delete_version(document_id, pipeline_version)

    def get_record(self, chunk_id: str) -> LayoutRetrievalRecord | None:
        row = self.connection.execute("SELECT record_json FROM chunks WHERE chunk_id=?", (chunk_id,)).fetchone()
        return _record_from_json(row[0]) if row else None

    def get_parent(self, parent_id: str, *, pipeline_version: str | None = None) -> LayoutSection | None:
        if pipeline_version is None:
            row = self.connection.execute(
                "SELECT record_json FROM parents WHERE parent_id=? ORDER BY pipeline_version DESC LIMIT 1",
                (parent_id,),
            ).fetchone()
        else:
            row = self.connection.execute(
                "SELECT record_json FROM parents WHERE parent_id=? AND pipeline_version=?",
                (parent_id, pipeline_version),
            ).fetchone()
        if not row:
            return None
        payload = json.loads(row[0])
        payload["region_ids"] = list(payload.get("region_ids") or [])
        return LayoutSection(**payload)

    def records_for_parent(
        self,
        parent_id: str,
        *,
        pipeline_version: str = LAYOUT_PIPELINE_VERSION,
        limit: int | None = None,
    ) -> tuple[LayoutRetrievalRecord, ...]:
        sql = "SELECT record_json FROM chunks WHERE parent_id=? AND pipeline_version=? AND indexable=1 ORDER BY reading_order"
        parameters: list[Any] = [parent_id, pipeline_version]
        if limit is not None:
            sql += " LIMIT ?"
            parameters.append(limit)
        return tuple(_record_from_json(row[0]) for row in self.connection.execute(sql, parameters))

    def chunk_ids_for_regions(
        self,
        region_ids: Sequence[str],
        *,
        pipeline_version: str = LAYOUT_PIPELINE_VERSION,
    ) -> tuple[str, ...]:
        if not region_ids:
            return ()
        placeholders = ",".join("?" for _ in region_ids)
        rows = self.connection.execute(
            f"SELECT DISTINCT cr.chunk_id FROM chunk_regions cr JOIN chunks c ON c.chunk_id=cr.chunk_id "
            f"WHERE cr.region_id IN ({placeholders}) AND c.pipeline_version=? AND c.indexable=1 ORDER BY c.reading_order",
            [*region_ids, pipeline_version],
        )
        return tuple(row[0] for row in rows)

    def chunk_ids_for_page(
        self,
        document_id: str,
        page_number: int,
        *,
        pipeline_version: str = LAYOUT_PIPELINE_VERSION,
    ) -> tuple[str, ...]:
        rows = self.connection.execute(
            "SELECT chunk_id FROM chunks WHERE document_id=? AND pipeline_version=? AND indexable=1 "
            "AND page_start<=? AND page_end>=? ORDER BY reading_order",
            (document_id, pipeline_version, page_number, page_number),
        )
        return tuple(row[0] for row in rows)

    def neighbors(self, record: LayoutRetrievalRecord, *, radius: int = 1) -> tuple[LayoutRetrievalRecord, ...]:
        return tuple(
            _record_from_json(row[0])
            for row in self.connection.execute(
                "SELECT record_json FROM chunks WHERE parent_id=? AND pipeline_version=? AND indexable=1 AND reading_order BETWEEN ? AND ? ORDER BY reading_order",
                (
                    record.parent_id,
                    record.pipeline_version,
                    record.reading_order - radius,
                    record.reading_order + radius,
                ),
            )
        )

    @staticmethod
    def _fts_phrase(query: str, *, max_characters: int = _DEFAULT_MAX_QUERY_CHARACTERS) -> str:
        cleaned = unicodedata.normalize("NFKC", query)[:max_characters].replace('"', " ").strip()
        return f'"{cleaned}"' if cleaned else ""

    @staticmethod
    def _fts_term_query(
        query: str,
        *,
        max_characters: int = _DEFAULT_MAX_QUERY_CHARACTERS,
        max_terms: int = _DEFAULT_MAX_QUERY_TERMS,
    ) -> str:
        normalized_query = _normalized_term(unicodedata.normalize("NFKC", query)[:max_characters])
        terms = [
            term.replace('"', " ").strip()
            for term in _query_terms(query, max_characters=max_characters, max_terms=max_terms)
            if term and term != normalized_query
        ]
        return " OR ".join(f'"{term}"' for term in dict.fromkeys(terms) if term)

    def search_lexical(
        self,
        query: str,
        *,
        pipeline_version: str = LAYOUT_PIPELINE_VERSION,
        limit: int = 30,
        max_query_characters: int = _DEFAULT_MAX_QUERY_CHARACTERS,
        max_query_terms: int = _DEFAULT_MAX_QUERY_TERMS,
    ) -> tuple[tuple[str, float], ...]:
        if not query.strip() or limit <= 0:
            return ()
        scores: dict[str, float] = {}
        for term in _query_terms(
            query,
            max_characters=max_query_characters,
            max_terms=max_query_terms,
        ):
            for row in self.connection.execute(
                "SELECT e.chunk_id FROM exact_terms e JOIN chunks c ON c.chunk_id=e.chunk_id WHERE e.term=? AND c.pipeline_version=? AND c.indexable=1",
                (term, pipeline_version),
            ):
                scores[row[0]] = scores.get(row[0], 0.0) + 4.0
        phrase = self._fts_phrase(query, max_characters=max_query_characters)
        term_query = self._fts_term_query(
            query,
            max_characters=max_query_characters,
            max_terms=max_query_terms,
        )
        fts_queries = tuple(dict.fromkeys(item for item in (phrase, term_query) if item))
        for query_index, fts_query in enumerate(fts_queries):
            try:
                rows = self.connection.execute(
                    "SELECT chunks_fts.chunk_id, bm25(chunks_fts,0.0,1.0,0.8,2.0,2.0) AS rank "
                    "FROM chunks_fts JOIN chunks ON chunks.chunk_id=chunks_fts.chunk_id "
                    "WHERE chunks_fts MATCH ? AND chunks.pipeline_version=? AND chunks.indexable=1 ORDER BY rank LIMIT ?",
                    (fts_query, pipeline_version, limit),
                ).fetchall()
            except sqlite3.OperationalError:
                rows = []
            for rank, row in enumerate(rows, 1):
                weight = 2.0 if query_index == 0 else 1.0
                scores[row[0]] = scores.get(row[0], 0.0) + weight / rank
        if not scores:
            escaped = unicodedata.normalize("NFKC", query)[:max_query_characters]
            escaped = escaped.replace("\\", "\\\\").replace("%", "\\%").replace("_", "\\_")
            like = f"%{escaped}%"
            for rank, row in enumerate(
                self.connection.execute(
                    "SELECT chunk_id FROM chunks WHERE pipeline_version=? AND indexable=1 "
                    "AND (exact_text LIKE ? ESCAPE '\\' OR section_path LIKE ? ESCAPE '\\') "
                    "ORDER BY reading_order LIMIT ?",
                    (pipeline_version, like, like, limit),
                ),
                1,
            ):
                scores[row[0]] = 1.0 / rank
        return tuple(sorted(scores.items(), key=lambda item: (-item[1], item[0]))[:limit])

    def search_parents(
        self,
        query: str,
        *,
        pipeline_version: str = LAYOUT_PIPELINE_VERSION,
        limit: int = 15,
        max_query_characters: int = _DEFAULT_MAX_QUERY_CHARACTERS,
        max_query_terms: int = _DEFAULT_MAX_QUERY_TERMS,
    ) -> tuple[tuple[str, float], ...]:
        if not query.strip() or limit <= 0:
            return ()
        phrase = self._fts_phrase(query, max_characters=max_query_characters)
        term_query = self._fts_term_query(
            query,
            max_characters=max_query_characters,
            max_terms=max_query_terms,
        )
        fts_query = term_query or phrase
        if not fts_query:
            return ()
        try:
            rows = self.connection.execute(
                "SELECT parent_id, bm25(parents_fts) AS rank FROM parents_fts WHERE parents_fts MATCH ? AND pipeline_version=? ORDER BY rank LIMIT ?",
                (fts_query, pipeline_version, limit),
            ).fetchall()
        except sqlite3.OperationalError:
            rows = []
        return tuple((row[0], 1.0 / rank) for rank, row in enumerate(rows, 1))

    def counts(self, document_id: str, *, pipeline_version: str = LAYOUT_PIPELINE_VERSION) -> dict[str, int]:
        return {
            table: int(
                self.connection.execute(
                    f"SELECT count(*) FROM {table} WHERE document_id=? AND pipeline_version=?",
                    (document_id, pipeline_version),
                ).fetchone()[0]
            )
            for table in ("pages", "regions", "parents", "chunks")
        }

    def close(self) -> None:
        self.connection.close()

    def __enter__(self) -> "SQLiteRetrievalRecordStore":
        return self

    def __exit__(self, *_args: object) -> None:
        self.close()

In [ ]:
# | export
def _scalar_metadata(value: Mapping[str, Any]) -> dict[str, str | int | float | bool]:
    metadata: dict[str, str | int | float | bool] = {}
    for key, item in value.items():
        if item is None:
            continue
        if isinstance(item, (str, int, float, bool)):
            metadata[key] = item
        else:
            metadata[key] = _json(item)
    return metadata


class ChromaDenseIndex:
    """Separate explicit-vector child and parent collections; never downloads an embedder."""

    def __init__(
        self,
        client: Any,
        *,
        pipeline_version: str = LAYOUT_PIPELINE_VERSION,
        embedding_model: str = "",
        collection_prefix: str = "layout_rag",
    ):
        if not embedding_model.strip():
            raise ValueError("embedding_model must identify the vector space")
        safe_prefix = re.sub(r"[^A-Za-z0-9_-]", "_", collection_prefix).strip("_-") or "layout_rag"
        safe_version = re.sub(r"[^A-Za-z0-9_-]", "_", pipeline_version).strip("_-") or "pipeline"
        namespace_digest = hashlib.sha256(
            f"{pipeline_version}\0{embedding_model}".encode("utf-8")
        ).hexdigest()[:16]
        namespace = f"{safe_prefix[:24]}_{safe_version[:24]}_{namespace_digest}"
        self.pipeline_version = pipeline_version
        self.embedding_model = embedding_model
        self.children = client.get_or_create_collection(
            name=f"{namespace}_children",
            embedding_function=None,
        )
        self.parents = client.get_or_create_collection(
            name=f"{namespace}_parents",
            embedding_function=None,
        )

    @staticmethod
    def _where(document_id: str, pipeline_version: str, embedding_model: str) -> dict[str, Any]:
        return {
            "$and": [
                {"document_id": document_id},
                {"pipeline_version": pipeline_version},
                {"embedding_model": embedding_model},
            ]
        }

    def _query_where(self) -> dict[str, Any]:
        return {
            "$and": [
                {"pipeline_version": self.pipeline_version},
                {"embedding_model": self.embedding_model},
            ]
        }

    @staticmethod
    def _snapshot(collection: Any, where: Mapping[str, Any]) -> dict[str, Any]:
        snapshot = collection.get(
            where=dict(where),
            include=["embeddings", "documents", "metadatas"],
        )
        ids = list(snapshot.get("ids") or [])
        raw_embeddings = snapshot.get("embeddings")
        embeddings = [] if raw_embeddings is None else [list(vector) for vector in raw_embeddings]
        raw_documents = snapshot.get("documents")
        documents = [] if raw_documents is None else list(raw_documents)
        raw_metadatas = snapshot.get("metadatas")
        metadatas = [] if raw_metadatas is None else list(raw_metadatas)
        return {
            "ids": ids,
            "embeddings": embeddings,
            "documents": documents,
            "metadatas": metadatas,
        }

    @staticmethod
    def _delete_where(collection: Any, where: Mapping[str, Any]) -> None:
        try:
            collection.delete(where=dict(where))
        except Exception as error:
            if "nothing found" not in str(error).casefold():
                raise

    @classmethod
    def _restore_snapshot(
        cls,
        collection: Any,
        where: Mapping[str, Any],
        snapshot: Mapping[str, Any],
    ) -> None:
        cls._delete_where(collection, where)
        ids = list(snapshot.get("ids") or [])
        if not ids:
            return
        collection.upsert(
            ids=ids,
            embeddings=list(snapshot.get("embeddings") or []),
            documents=list(snapshot.get("documents") or []),
            metadatas=list(snapshot.get("metadatas") or []),
        )

    def delete_document(self, document_id: str) -> None:
        where = self._where(document_id, self.pipeline_version, self.embedding_model)
        for collection in (self.children, self.parents):
            self._delete_where(collection, where)

    def replace(
        self,
        result: LayoutIngestionResult,
        *,
        child_embeddings: Sequence[Sequence[float]],
        parent_embeddings: Sequence[Sequence[float]],
    ) -> None:
        if result.document.pipeline_version != self.pipeline_version:
            raise ValueError(
                f"result pipeline version {result.document.pipeline_version!r} does not match dense index {self.pipeline_version!r}"
            )
        children = [record for record in result.records if record.indexable]
        parents = [parent for parent in result.sections if parent.summary and not parent.is_toc]
        if len(children) != len(child_embeddings):
            raise ValueError("child embedding count does not match indexable records")
        if len(parents) != len(parent_embeddings):
            raise ValueError("parent embedding count does not match indexable parents")
        child_vectors = self._validated_vectors(child_embeddings, "child")
        parent_vectors = self._validated_vectors(parent_embeddings, "parent")
        where = self._where(result.document.document_id, self.pipeline_version, self.embedding_model)
        child_snapshot = self._snapshot(self.children, where)
        parent_snapshot = self._snapshot(self.parents, where)
        old_child_ids = set(child_snapshot["ids"])
        old_parent_ids = set(parent_snapshot["ids"])
        child_ids = [record.chunk_id for record in children]
        parent_ids = [parent.parent_id for parent in parents]
        try:
            if children:
                self.children.upsert(
                    ids=child_ids,
                    embeddings=child_vectors,
                    documents=[record.retrieval_text for record in children],
                    metadatas=[
                        _scalar_metadata(
                            {
                                "document_id": record.document_id,
                                "pipeline_version": record.pipeline_version,
                                "parent_id": record.parent_id,
                                "content_type": record.content_type,
                                "page_start": record.page_start,
                                "page_end": record.page_end,
                                "embedding_model": self.embedding_model,
                            }
                        )
                        for record in children
                    ],
                )
            if parents:
                self.parents.upsert(
                    ids=parent_ids,
                    embeddings=parent_vectors,
                    documents=[parent.summary for parent in parents],
                    metadatas=[
                        _scalar_metadata(
                            {
                                "document_id": parent.document_id,
                                "pipeline_version": result.document.pipeline_version,
                                "section_path": parent.section_path,
                                "page_start": parent.page_start,
                                "page_end": parent.page_end,
                                "embedding_model": self.embedding_model,
                            }
                        )
                        for parent in parents
                    ],
                )
            stale_children = sorted(old_child_ids - set(child_ids))
            stale_parents = sorted(old_parent_ids - set(parent_ids))
            if stale_children:
                self.children.delete(ids=stale_children)
            if stale_parents:
                self.parents.delete(ids=stale_parents)
        except Exception as error:
            rollback_errors: list[Exception] = []
            for collection, snapshot in (
                (self.children, child_snapshot),
                (self.parents, parent_snapshot),
            ):
                try:
                    self._restore_snapshot(collection, where, snapshot)
                except Exception as rollback_error:
                    rollback_errors.append(rollback_error)
            if rollback_errors:
                raise RuntimeError(
                    f"dense replacement failed and rollback also failed: {rollback_errors[0]}"
                ) from error
            raise

    @staticmethod
    def _validated_vectors(
        vectors: Sequence[Sequence[float]],
        label: str,
    ) -> list[list[float]]:
        normalized = [[float(value) for value in vector] for vector in vectors]
        dimensions = {len(vector) for vector in normalized}
        if normalized and (0 in dimensions or len(dimensions) != 1):
            raise ValueError(f"{label} embeddings must be nonempty and have one dimension")
        if any(not math.isfinite(value) for vector in normalized for value in vector):
            raise ValueError(f"{label} embeddings contain non-finite values")
        return normalized

    @staticmethod
    def _query(
        collection: Any,
        vector: Sequence[float],
        limit: int,
        *,
        where: Mapping[str, Any],
    ) -> tuple[tuple[str, float], ...]:
        if limit <= 0:
            return ()
        try:
            result = collection.query(
                query_embeddings=[list(vector)],
                n_results=limit,
                where=dict(where),
                include=["distances"],
            )
        except Exception as error:
            if "empty" in str(error).casefold() or "nothing" in str(error).casefold():
                return ()
            raise
        ids = (result.get("ids") or [[]])[0]
        distances = (result.get("distances") or [[]])[0]
        return tuple(
            (identifier, 1.0 / (1.0 + max(0.0, float(distance))))
            for identifier, distance in zip(ids, distances)
        )

    def query_children(self, query_embedding: Sequence[float], *, limit: int) -> tuple[tuple[str, float], ...]:
        return self._query(self.children, query_embedding, limit, where=self._query_where())

    def query_parents(self, query_embedding: Sequence[float], *, limit: int) -> tuple[tuple[str, float], ...]:
        return self._query(self.parents, query_embedding, limit, where=self._query_where())


class OllamaEmbeddingProvider:
    """Small adapter that keeps the embedding model separate from generation/vision models."""

    def __init__(self, client: Any, model: str = "bge-m3"):
        self.client = client
        self.model = model

    async def __call__(self, texts: Sequence[str]) -> Sequence[Sequence[float]]:
        response = await self.client.embed(model=self.model, input=list(texts))
        embeddings = response.get("embeddings") if isinstance(response, Mapping) else getattr(response, "embeddings", None)
        if not isinstance(embeddings, Sequence) or len(embeddings) != len(texts):
            raise ValueError("embedding provider returned an invalid batch")
        return embeddings


async def _embed(provider: EmbeddingProvider, texts: Sequence[str]) -> Sequence[Sequence[float]]:
    result = provider(texts)
    return await result if inspect.isawaitable(result) else result


class HybridIndexer:
    """Validate/compute embeddings before replacing lexical and optional dense state."""

    def __init__(self, store: SQLiteRetrievalRecordStore, dense_index: ChromaDenseIndex | None = None):
        self.store = store
        self.dense_index = dense_index

    async def index(
        self,
        result: LayoutIngestionResult,
        *,
        embedder: EmbeddingProvider | None = None,
        child_embeddings: Sequence[Sequence[float]] | None = None,
        parent_embeddings: Sequence[Sequence[float]] | None = None,
    ) -> None:
        _validate_ingestion_result(result)
        if self.dense_index is None:
            self.store.replace(result)
            return
        children = [record for record in result.records if record.indexable]
        parents = [parent for parent in result.sections if parent.summary and not parent.is_toc]
        if child_embeddings is None or parent_embeddings is None:
            if embedder is None:
                raise ValueError("dense indexing requires an embedder or explicit child and parent embeddings")
            child_embeddings = await _embed(embedder, [record.retrieval_text for record in children])
            parent_embeddings = await _embed(embedder, [parent.summary for parent in parents])
        self.dense_index._validated_vectors(child_embeddings, "child")
        self.dense_index._validated_vectors(parent_embeddings, "parent")
        if result.document.pipeline_version != self.dense_index.pipeline_version:
            raise ValueError("result and dense index pipeline versions differ")
        with self.store.connection:
            self.store.replace(result, commit=False)
            self.dense_index.replace(
                result,
                child_embeddings=child_embeddings,
                parent_embeddings=parent_embeddings,
            )

In [ ]:
# | export
class EvidenceExpander:
    """Expand precise child hits only within their instruction parent."""

    def __init__(self, store: SQLiteRetrievalRecordStore, *, neighbor_radius: int = 1):
        self.store = store
        self.neighbor_radius = neighbor_radius

    def expand(self, hit: SearchHit) -> Evidence:
        records = self.store.neighbors(hit.record, radius=self.neighbor_radius)
        parent = self.store.get_parent(hit.record.parent_id, pipeline_version=hit.record.pipeline_version)
        citation = CitationAssembler.format_records(records or (hit.record,))
        return Evidence(hit=hit, parent=parent, records=records or (hit.record,), citation=citation)


class HybridRetriever:
    """Fuse exact/FTS, dense child/parent, and optional visual candidates with RRF."""

    def __init__(
        self,
        store: SQLiteRetrievalRecordStore,
        *,
        dense_index: DenseCandidateIndex | None = None,
        reranker: Reranker | None = None,
        visual_retriever: VisualRetriever | None = None,
        config: LayoutRAGConfig | None = None,
    ):
        self.store = store
        self.dense_index = dense_index
        self.reranker = reranker
        self.visual_retriever = visual_retriever
        self.config = config or LayoutRAGConfig()
        self.expander = EvidenceExpander(store, neighbor_radius=self.config.neighbor_radius)

    @staticmethod
    def _ids(candidates: Sequence[str] | Sequence[tuple[str, float]]) -> list[str]:
        return [candidate if isinstance(candidate, str) else candidate[0] for candidate in candidates]

    def _support_score(self, query: str, record: LayoutRetrievalRecord) -> int:
        normalized_text = _normalized_term(record.exact_text)
        return sum(
            1
            for term in _query_terms(
                query,
                max_characters=self.config.max_query_characters,
                max_terms=self.config.max_query_terms,
            )
            if term and term in normalized_text
        )

    def _parent_candidates(
        self,
        query: str,
        parent_ids: Sequence[str],
        *,
        preferred_child_ids: Sequence[str] = (),
    ) -> list[str]:
        chunk_ids: list[str] = []
        preferred_ranks = {chunk_id: rank for rank, chunk_id in enumerate(preferred_child_ids)}
        for parent_id in parent_ids:
            records = self.store.records_for_parent(
                parent_id,
                pipeline_version=self.config.pipeline_version,
            )
            supported = sorted(
                (
                    (
                        self._support_score(query, record),
                        preferred_ranks.get(record.chunk_id, math.inf),
                        record,
                    )
                    for record in records
                ),
                key=lambda item: (-item[0], item[1], item[2].reading_order),
            )
            direct = [
                record.chunk_id
                for score, preferred_rank, record in supported
                if score > 0 or math.isfinite(preferred_rank)
            ]
            chunk_ids.extend((direct or [record.chunk_id for record in records[:1]])[:2])
        return chunk_ids

    def _visual_candidates(
        self,
        query: str,
        candidates: Sequence[str] | Sequence[tuple[str, float]],
    ) -> list[str]:
        chunk_ids: list[str] = []
        for identifier in self._ids(candidates):
            record = self.store.get_record(identifier)
            if record is not None and record.pipeline_version == self.config.pipeline_version:
                chunk_ids.append(identifier)
                continue
            region_id = identifier.removesuffix(":crop")
            mapped = self.store.chunk_ids_for_regions(
                [region_id],
                pipeline_version=self.config.pipeline_version,
            )
            if mapped:
                chunk_ids.extend(mapped)
                continue
            page_match = re.match(r"(?P<document>.+):page-image:(?P<page>\d+):\d+dpi$", identifier)
            if page_match:
                page_chunk_ids = self.store.chunk_ids_for_page(
                    page_match.group("document"),
                    int(page_match.group("page")),
                    pipeline_version=self.config.pipeline_version,
                )
                page_records = [self.store.get_record(chunk_id) for chunk_id in page_chunk_ids]
                page_records = [record for record in page_records if record is not None]
                visual_query = bool(_VISUAL_QUERY_RE.search(query))
                typed_visual_records = [
                    record
                    for record in page_records
                    if record.content_type in {"figure", "table_rows", "formula"}
                ]
                if visual_query and typed_visual_records:
                    page_records = typed_visual_records
                page_records.sort(
                    key=lambda record: (
                        -self._support_score(query, record),
                        -(int(record.content_type in {"figure", "table_rows", "formula"}) if visual_query else 0),
                        record.reading_order,
                    )
                )
                chunk_ids.extend(
                    record.chunk_id
                    for record in page_records[: self.config.max_visual_chunks_per_asset]
                )
        return chunk_ids

    def retrieve(
        self,
        query: str,
        *,
        query_embedding: Sequence[float] | None = None,
        limit: int | None = None,
    ) -> tuple[SearchHit, ...]:
        config = self.config
        if not query.strip():
            return ()
        bounded_query = unicodedata.normalize("NFKC", query)[: config.max_query_characters]
        result_limit = config.result_limit if limit is None else limit
        if result_limit <= 0:
            return ()
        channels: dict[str, list[str]] = {}
        channels["lexical"] = self._ids(
            self.store.search_lexical(
                bounded_query,
                pipeline_version=config.pipeline_version,
                limit=config.lexical_top_k,
                max_query_characters=config.max_query_characters,
                max_query_terms=config.max_query_terms,
            )
        )
        lexical_parents = self._ids(
            self.store.search_parents(
                bounded_query,
                pipeline_version=config.pipeline_version,
                limit=config.dense_parent_top_k,
                max_query_characters=config.max_query_characters,
                max_query_terms=config.max_query_terms,
            )
        )
        if lexical_parents:
            channels["lexical_parent"] = self._parent_candidates(bounded_query, lexical_parents)
        if self.dense_index is not None and query_embedding is not None:
            dense_children = self._ids(
                self.dense_index.query_children(query_embedding, limit=config.dense_child_top_k)
            )
            channels["dense_child"] = dense_children
            dense_parents = self._ids(
                self.dense_index.query_parents(query_embedding, limit=config.dense_parent_top_k)
            )
            channels["dense_parent"] = self._parent_candidates(
                bounded_query,
                dense_parents,
                preferred_child_ids=dense_children,
            )
        if self.visual_retriever is not None:
            channels["visual"] = self._visual_candidates(
                bounded_query,
                self.visual_retriever(bounded_query, config.visual_top_k),
            )

        lexical_weight = config.lexical_weight * (
            config.syntax_lexical_boost if _SYNTAX_QUERY_RE.search(bounded_query) else 1.0
        )
        weights = {
            "lexical": lexical_weight,
            "lexical_parent": lexical_weight * config.dense_parent_weight,
            "dense_child": config.dense_child_weight,
            "dense_parent": config.dense_parent_weight,
            "visual": config.visual_weight,
        }
        scores: dict[str, float] = {}
        ranks: dict[str, dict[str, int]] = defaultdict(dict)
        for channel, identifiers in channels.items():
            for rank, chunk_id in enumerate(dict.fromkeys(identifiers), 1):
                scores[chunk_id] = scores.get(chunk_id, 0.0) + weights[channel] / (config.rrf_k + rank)
                ranks[chunk_id][channel] = rank
        hits: list[SearchHit] = []
        for chunk_id, score in scores.items():
            record = self.store.get_record(chunk_id)
            if record is not None and record.indexable:
                if record.pipeline_version != config.pipeline_version:
                    continue
                adjusted_score = score
                if record.content_type == "figure" and "visual" not in ranks[chunk_id] and not _VISUAL_QUERY_RE.search(bounded_query):
                    adjusted_score *= config.text_query_figure_weight
                hits.append(SearchHit(record=record, score=adjusted_score, channel_ranks=ranks[chunk_id]))
        hits.sort(key=lambda hit: (-hit.score, hit.record.reading_order, hit.record.chunk_id))
        hits = hits[: config.rerank_top_k]
        if self.reranker is not None and hits:
            reranked = self.reranker(bounded_query, hits)
            if isinstance(reranked, Mapping):
                for hit in hits:
                    if hit.record.chunk_id in reranked:
                        hit.rerank_score = float(reranked[hit.record.chunk_id])
                hits.sort(
                    key=lambda hit: (
                        -(hit.rerank_score if hit.rerank_score is not None else -math.inf),
                        -hit.score,
                        hit.record.chunk_id,
                    )
                )
            else:
                allowed = {hit.record.chunk_id: hit for hit in hits}
                safe_hits: list[SearchHit] = []
                seen: set[str] = set()
                for reranked_hit in reranked:
                    chunk_id = reranked_hit.record.chunk_id
                    if chunk_id in seen or chunk_id not in allowed:
                        continue
                    original = allowed[chunk_id]
                    if not original.record.indexable:
                        continue
                    original.rerank_score = reranked_hit.rerank_score
                    safe_hits.append(original)
                    seen.add(chunk_id)
                hits = safe_hits

        selected: list[SearchHit] = []
        parent_counts: dict[str, int] = defaultdict(int)
        page_counts: dict[tuple[str, int], int] = defaultdict(int)
        for hit in hits:
            if parent_counts[hit.record.parent_id] >= config.max_hits_per_parent:
                continue
            hit_pages = range(hit.record.page_start, hit.record.page_end + 1)
            if any(page_counts[(hit.record.document_id, page)] >= config.max_hits_per_page for page in hit_pages):
                continue
            selected.append(hit)
            parent_counts[hit.record.parent_id] += 1
            for page in hit_pages:
                page_counts[(hit.record.document_id, page)] += 1
            if len(selected) >= result_limit:
                break
        return tuple(selected)

    def retrieve_evidence(
        self,
        query: str,
        *,
        query_embedding: Sequence[float] | None = None,
        limit: int | None = None,
    ) -> tuple[Evidence, ...]:
        return tuple(
            self.expander.expand(hit)
            for hit in self.retrieve(query, query_embedding=query_embedding, limit=limit)
        )

In [ ]:
# | export
class CitationAssembler:
    """Build compact citations that retain page, region, bbox, and source evidence."""

    @staticmethod
    def _region_index(region_id: str) -> int:
        match = re.search(r":r(\d+)$", region_id)
        return int(match.group(1)) if match else 0

    @staticmethod
    def _region_location(region_id: str) -> tuple[int, int] | None:
        match = re.search(r":p(\d+):r(\d+)$", region_id)
        return (int(match.group(1)), int(match.group(2))) if match else None

    @staticmethod
    def _format_indices(indices: Sequence[int]) -> str:
        unique = sorted(set(indices))
        if len(unique) == 1:
            return f"region {unique[0]}"
        if unique == list(range(unique[0], unique[-1] + 1)):
            return f"regions {unique[0]}–{unique[-1]}"
        return "regions " + ",".join(str(index) for index in unique)

    @classmethod
    def format_records(cls, records: Sequence[LayoutRetrievalRecord]) -> str:
        if not records:
            return ""
        first = records[0]
        by_page: dict[int, list[int]] = defaultdict(list)
        for record in records:
            for region_id in record.region_ids:
                location = cls._region_location(region_id)
                if location is not None:
                    by_page[location[0]].append(location[1])
        locations = "; ".join(
            f"p.{page} {cls._format_indices(indices)}"
            for page, indices in sorted(by_page.items())
        )
        if not locations:
            pages = sorted({page for record in records for page in range(record.page_start, record.page_end + 1)})
            locations = f"p.{pages[0]}" if len(pages) == 1 else f"pp.{pages[0]}–{pages[-1]}"
        identity = " ".join(item for item in (first.document_code, first.revision) if item) or first.document_title
        return f"[{identity}, {locations}]"

    @staticmethod
    def citation_payload(evidence: Evidence) -> dict[str, Any]:
        evidence_items: dict[str, dict[str, Any]] = {}
        for record in evidence.records:
            for index, region_id in enumerate(record.region_ids):
                location = CitationAssembler._region_location(region_id)
                evidence_items.setdefault(
                    region_id,
                    {
                        "page": location[0] if location else record.page_start,
                        "region_id": region_id,
                        "bbox": record.bboxes[index] if index < len(record.bboxes) else None,
                        "bbox_normalized": record.bboxes_normalized[index]
                        if index < len(record.bboxes_normalized)
                        else None,
                        "ocr_status": record.ocr_statuses[index]
                        if index < len(record.ocr_statuses)
                        else None,
                        "text_source": record.text_sources[index]
                        if index < len(record.text_sources)
                        else None,
                        "exact_text": record.canonical_texts[index]
                        if index < len(record.canonical_texts)
                        else record.exact_text,
                        "cleaned_ocr": record.cleaned_ocr_texts[index]
                        if index < len(record.cleaned_ocr_texts)
                        else None,
                        "native_text": record.native_texts[index]
                        if index < len(record.native_texts)
                        else None,
                        "quality_flags": list(record.region_quality_flags[index])
                        if index < len(record.region_quality_flags)
                        else [],
                        "asset_path": record.region_asset_paths[index]
                        if index < len(record.region_asset_paths)
                        else None,
                    },
                )
        source_hashes = dict(evidence.hit.record.source_hashes)
        return {
            "citation": evidence.citation,
            "document_id": evidence.hit.record.document_id,
            "document_code": evidence.hit.record.document_code,
            "revision": evidence.hit.record.revision,
            "pages": sorted({page for record in evidence.records for page in range(record.page_start, record.page_end + 1)}),
            "evidence": list(evidence_items.values()),
            "region_ids": list(evidence_items),
            "bboxes": [item["bbox"] for item in evidence_items.values()],
            "bboxes_normalized": [item["bbox_normalized"] for item in evidence_items.values()],
            "source_pdf_uri": evidence.hit.record.source_pdf_uri,
            "source_markdown_uri": evidence.hit.record.source_markdown_uri,
            "source_layout_uri": evidence.hit.record.source_layout_uri,
            "source_hashes": source_hashes,
            "asset_paths": list(
                dict.fromkeys(path for record in evidence.records for path in record.asset_paths)
            ),
            "quality_flags": list(
                dict.fromkeys(flag for record in evidence.records for flag in record.quality_flags)
            ),
            "pipeline_version": evidence.hit.record.pipeline_version,
            "exact_text": evidence.exact_text,
        }


@dataclass(frozen=True)
class RetrievalEvalCase:
    query: str
    relevant_chunk_ids: tuple[str, ...] = ()
    relevant_parent_ids: tuple[str, ...] = ()
    relevant_pages: tuple[int, ...] = ()
    query_embedding: tuple[float, ...] | None = None
    expected_answerable: bool | None = None


@dataclass(frozen=True)
class RetrievalMetrics:
    cases: int
    answerable_cases: int
    unanswerable_cases: int
    child_recall_at_k: float
    parent_recall_at_k: float
    page_recall_at_k: float
    mean_reciprocal_rank: float
    abstention_accuracy: float


def evaluate_retriever(
    retriever: HybridRetriever,
    cases: Sequence[RetrievalEvalCase],
    *,
    k: int = 8,
) -> RetrievalMetrics:
    """Measure deterministic child/parent/page recall and reciprocal rank."""
    if not cases:
        return RetrievalMetrics(0, 0, 0, 0.0, 0.0, 0.0, 0.0, 0.0)
    child_hits = parent_hits = page_hits = 0
    child_cases = parent_cases = page_cases = 0
    answerable_cases = unanswerable_cases = abstentions = 0
    reciprocal_ranks: list[float] = []
    for case in cases:
        hits = retriever.retrieve(case.query, query_embedding=case.query_embedding, limit=k)
        chunk_ids = [hit.record.chunk_id for hit in hits]
        parent_ids = {hit.record.parent_id for hit in hits}
        pages = {page for hit in hits for page in range(hit.record.page_start, hit.record.page_end + 1)}
        answerable = case.expected_answerable
        if answerable is None:
            answerable = bool(case.relevant_chunk_ids or case.relevant_parent_ids or case.relevant_pages)
        if answerable:
            answerable_cases += 1
        else:
            unanswerable_cases += 1
            abstentions += int(not hits)
        if case.relevant_chunk_ids:
            child_cases += 1
            child_hits += int(bool(set(case.relevant_chunk_ids) & set(chunk_ids)))
        if case.relevant_parent_ids:
            parent_cases += 1
            parent_hits += int(bool(set(case.relevant_parent_ids) & parent_ids))
        if case.relevant_pages:
            page_cases += 1
            page_hits += int(bool(set(case.relevant_pages) & pages))
        relevant = set(case.relevant_chunk_ids)
        if relevant:
            rank = next((position for position, chunk_id in enumerate(chunk_ids, 1) if chunk_id in relevant), None)
            reciprocal_ranks.append(1.0 / rank if rank else 0.0)
    count = len(cases)
    return RetrievalMetrics(
        cases=count,
        answerable_cases=answerable_cases,
        unanswerable_cases=unanswerable_cases,
        child_recall_at_k=child_hits / child_cases if child_cases else 0.0,
        parent_recall_at_k=parent_hits / parent_cases if parent_cases else 0.0,
        page_recall_at_k=page_hits / page_cases if page_cases else 0.0,
        mean_reciprocal_rank=sum(reciprocal_ranks) / len(reciprocal_ranks) if reciprocal_ranks else 0.0,
        abstention_accuracy=abstentions / unanswerable_cases if unanswerable_cases else 0.0,
    )

In [ ]:
# | hide
# | notest
import nbdev

nbdev.nbdev_export()